# ДЗ-2. Генерация кода — итоговая V2

В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Выполните ячейки сверху вниз. Загрузить нужно только `HW2_CODE_FINAL.zip`;
веса, токенизатор и все 1919 тестовых входов уже находятся внутри.

Параметры зафиксированы: шаг 111275, FP16, greedy, batch size 32,
максимум 384 новых токена. Используется исходный генератор.


In [ ]:
import sys
import json
import subprocess
import importlib.metadata
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Выберите T4 GPU в настройках среды Colab'
gpu = torch.cuda.get_device_name(0)
assert 'T4' in gpu, f'Для контрольного запуска нужна NVIDIA T4, получена {gpu}'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', gpu)

try:
    tokenizers_version = importlib.metadata.version('tokenizers')
except importlib.metadata.PackageNotFoundError:
    tokenizers_version = None
if tokenizers_version != '0.22.2':
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', 'tokenizers==0.22.2'
    ], check=True)
print('tokenizers:', importlib.metadata.version('tokenizers'))


## Загрузка проекта

Если проект уже распакован, ячейка найдёт его автоматически.
Иначе выберите `HW2_CODE_FINAL.zip` в появившемся окне загрузки.
Датасет скачивать и обучение запускать не требуется.


In [ ]:
import os
import zipfile

base = Path('/content') if Path('/content').is_dir() else Path.cwd()
candidates = [Path.cwd(), Path.cwd().parent, base / 'hw2_code_generation']
project = next((p.resolve() for p in candidates
                if (p / 'generate.py').is_file()
                and (p / 'model_weights/t4_step111275.zip').is_file()), None)
if project is None:
    archives = [base / 'HW2_CODE_FINAL.zip', Path.cwd() / 'HW2_CODE_FINAL.zip']
    archive = next((p for p in archives if p.is_file()), None)
    if archive is None:
        from google.colab import files
        os.chdir(base)
        uploaded = files.upload()
        archive = next((base / name for name in uploaded
                        if name == 'HW2_CODE_FINAL.zip'), None)
        del uploaded
    assert archive is not None and archive.is_file(), 'Загрузите HW2_CODE_FINAL.zip'
    with zipfile.ZipFile(archive) as z:
        assert all(not Path(name).is_absolute() and '..' not in Path(name).parts
                   and Path(name).parts[0] == 'hw2_code_generation'
                   for name in z.namelist()), 'Неожиданная структура ZIP'
        z.extractall(base)
    project = base / 'hw2_code_generation'
assert (project / 'generate.py').is_file()
print('Проект:', project)


## Полный инференс

Выполняются три полных прохода. Время каждого включает сортировку,
padding, перенос данных, encoder/decoder, detokenization и запись JSONL.
Загрузка весов и прогрев измеряются отдельно.

Каждый новый результат сравнивается **побайтно** с приложенной посылкой,
для которой зафиксирован public pass@1 74,27%. При расхождении или
превышении 180 секунд выполнение завершится с ошибкой.


In [ ]:
output = project / 'outputs'
subprocess.run([
    sys.executable, str(project / 'generate.py'),
    '--output', str(output), '--repeats', '3',
], check=True)

benchmark = json.loads((output / 't4_benchmark.json').read_text(encoding='utf-8'))
print('Полные проходы, сек:', benchmark['full_run_seconds'])
print('Загрузка весов, сек:', benchmark['model_load_seconds'])
print('Прогрев, сек:', benchmark['warmup_seconds'])
print('Загрузка + прогрев + первый проход, сек:',
      benchmark['load_plus_warmup_plus_first_full_seconds'])
assert benchmark['within_180_seconds']
assert benchmark['repeated_outputs_identical']


In [ ]:
import hashlib

submission = output / 'submission.jsonl'
answers = [json.loads(line) for line in submission.read_text(encoding='utf-8').splitlines()]
assert len(answers) == 1919
assert all(isinstance(row, dict) and set(row) == {'code'}
           and isinstance(row['code'], str) for row in answers)
assert submission.read_bytes() == (project / 'submission.jsonl').read_bytes()
print('Строк:', len(answers), 'Порядок: исходный test')
print('SHA-256:', hashlib.sha256(submission.read_bytes()).hexdigest())
print('Результат точно совпал с отправленной посылкой.')

if Path('/content').is_dir():
    from google.colab import files
    files.download(str(submission))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(submission)))
